In [11]:
import os
print(os.getcwd())


/Users/sohamkhopkar/Desktop/Resume/Projects/Finance Analytics/notebooks


In [14]:
import pandas as pd
import numpy as np
from pathlib import Path

In [13]:
df = pd.read_csv("../data/raw/dataset.csv",nrows=10000)

In [15]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [16]:
df.shape

(10000, 11)

In [18]:
df.columns.tolist()

['step',
 'type',
 'amount',
 'nameOrig',
 'oldbalanceOrg',
 'newbalanceOrig',
 'nameDest',
 'oldbalanceDest',
 'newbalanceDest',
 'isFraud',
 'isFlaggedFraud']

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   step            10000 non-null  int64  
 1   type            10000 non-null  object 
 2   amount          10000 non-null  float64
 3   nameOrig        10000 non-null  object 
 4   oldbalanceOrg   10000 non-null  float64
 5   newbalanceOrig  10000 non-null  float64
 6   nameDest        10000 non-null  object 
 7   oldbalanceDest  10000 non-null  float64
 8   newbalanceDest  10000 non-null  float64
 9   isFraud         10000 non-null  int64  
 10  isFlaggedFraud  10000 non-null  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 859.5+ KB


In [21]:
df.isnull().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

In [22]:
df["type"].value_counts()

type
PAYMENT     5465
CASH_IN     1949
CASH_OUT    1321
TRANSFER     921
DEBIT        344
Name: count, dtype: int64

In [23]:
df["isFraud"].value_counts()

isFraud
0    9932
1      68
Name: count, dtype: int64

In [24]:
df["amount"].describe()

count    1.000000e+04
mean     1.035467e+05
std      2.663072e+05
min      2.390000e+00
25%      4.397530e+03
50%      1.285874e+04
75%      1.143825e+05
max      1.000000e+07
Name: amount, dtype: float64

In [27]:
total_transactions = len(df)
total_amount = df["amount"].sum()
average_amount = df["amount"].mean()
fraud_count = df["isFraud"].sum()
fraud_rate = df["isFraud"].mean()* 100
flagged_count = df["isFlaggedFraud"].sum()

In [30]:
print(f"Total transactions: {total_transactions:,}")
print(f"Total transaction value: {total_amount:,.2f}")
print(f"Average transaction amount: {average_amount:,.2f}")
print(f"Fraudulent transactions: {fraud_count:,}")
print(f"Fraud rate: {fraud_rate:.2f}%")
print(f"Transactions flagged by system: {flagged_count:,}")

Total transactions: 10,000
Total transaction value: 1,035,466,899.49
Average transaction amount: 103,546.69
Fraudulent transactions: 68
Fraud rate: 0.68%
Transactions flagged by system: 0


In [32]:
print("Duplicate rows",df.duplicated().sum())

Duplicate rows 0


In [33]:
print("Negative amount", (df["amount"]<0).sum())

Negative amount 0


In [38]:
balance_columns = [
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]
print("Negative Balance\n",(df[balance_columns]<0).sum())

Negative Balance
 oldbalanceOrg     0
newbalanceOrig    0
oldbalanceDest    0
newbalanceDest    0
dtype: int64


In [42]:
print("Fraud Label values\n",sorted(df["isFraud"].unique()))

Fraud Label values
 [np.int64(0), np.int64(1)]


In [46]:
print("Transaction Types")
print(sorted(df["type"].unique()))

Transaction Types
['CASH_IN', 'CASH_OUT', 'DEBIT', 'PAYMENT', 'TRANSFER']


In [48]:
df_clean = df.copy()

Calculating the difference between the original and new sender balance

In [49]:
df_clean["balance_diff"] = (
    df_clean["oldbalanceOrg"] - df_clean["newbalanceOrig"]
)

In [52]:
high_value_threshold = df_clean["amount"].quantile(0.90)
df_clean["is_high_value"] = (
    df_clean["amount"] > high_value_threshold
).astype(int)



In [53]:
df_clean["fraud_status"] = df_clean["isFraud"].map({
    0: "Legitimate",
    1: "Fraud"
})

In [54]:
print("High-value threshold", round(high_value_threshold, 2))


High-value threshold 284452.18


In [55]:
df_clean[
    ["amount", "balance_diff", "is_high_value", "fraud_status"]
].head()

,amount,balance_diff,is_high_value,fraud_status
0,9839.64,9839.64,0,Legitimate
1,1864.28,1864.28,0,Legitimate
2,181.00,181.00,0,Fraud
3,181.00,181.00,0,Fraud
4,11668.14,11668.14,0,Legitimate


In [56]:
fraud_summary = (
    df_clean.groupby("type")
    .agg(
        transaction_count=("amount", "count"),
        total_value=("amount", "sum"),
        average_amount=("amount", "mean"),
        fraud_count=("isFraud", "sum")
    )
    .reset_index()
)


In [57]:
fraud_summary["fraud_rate_pct"] = (
    fraud_summary["fraud_count"]
    / fraud_summary["transaction_count"]
    * 100
)

In [58]:
fraud_summary.sort_values(
    "fraud_rate_pct",
    ascending=False
)

,type,transaction_count,total_value,average_amount,fraud_count,fraud_rate_pct
4,TRANSFER,921,4.343938e+08,471654.518914,33,3.583062
1,CASH_OUT,1321,2.292032e+08,173507.343543,35,2.649508
0,CASH_IN,1949,3.275129e+08,168041.492468,0,0.000000
2,DEBIT,344,1.230348e+06,3576.593721,0,0.000000
3,PAYMENT,5465,4.312667e+07,7891.430867,0,0.000000


In [59]:
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

In [60]:
output_path = output_dir / "transactions_cleaned.csv"

df_clean.to_csv(output_path, index=False)

print(f"Cleaned data saved to: {output_path}")
print(f"Rows saved: {len(df_clean):,}")

Cleaned data saved to: ../data/processed/transactions_cleaned.csv
Rows saved: 10,000


In [61]:
def clean_transactions(data):

    cleaned = data.copy()

    cleaned.columns = cleaned.columns.str.strip()

    cleaned = cleaned.drop_duplicates()

    numeric_columns = [
        "step", "amount", "oldbalanceOrg",
        "newbalanceOrig", "oldbalanceDest",
        "newbalanceDest", "isFraud", "isFlaggedFraud"
    ]

    for column in numeric_columns:
        cleaned[column] = pd.to_numeric(
            cleaned[column], errors="raise"
        )

    if (cleaned["amount"] < 0).any():
        raise ValueError("Negative transaction amounts found.")

    balance_columns = [
        "oldbalanceOrg", "newbalanceOrig",
        "oldbalanceDest", "newbalanceDest"
    ]

    if (cleaned[balance_columns] < 0).any().any():
        raise ValueError("Negative balances found.")

    if not cleaned["isFraud"].isin([0, 1]).all():
        raise ValueError("Unexpected fraud labels found.")

    cleaned["balance_diff"] = (
        cleaned["oldbalanceOrg"] - cleaned["newbalanceOrig"]
    )

    threshold = cleaned["amount"].quantile(0.90)
    cleaned["is_high_value"] = (
        cleaned["amount"] > threshold
    ).astype(int)

    cleaned["fraud_status"] = cleaned["isFraud"].map({
        0: "Legitimate",
        1: "Fraud"
    })

    return cleaned

In [62]:
df_clean = clean_transactions(df)

print("Original rows", len(df))
print("Cleaned rows", len(df_clean))
print("Fraudulent transactions", int(df_clean["isFraud"].sum()))

df_clean.head()

Original rows 10000
Cleaned rows 10000
Fraudulent transactions 68


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,balance_diff,is_high_value,fraud_status
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0,9839.64,0,Legitimate
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0,1864.28,0,Legitimate
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0,181.00,0,Fraud
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0,181.00,0,Fraud
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0,11668.14,0,Legitimate


In [67]:
%pip install "psycopg[binary]"

  Using cached psycopg-3.3.6-py3-none-any.whl.metadata (4.4 kB)
  Using cached psycopg_binary-3.3.6-cp312-cp312-macosx_11_0_arm64.whl.metadata (2.7 kB)
Using cached psycopg_binary-3.3.6-cp312-cp312-macosx_11_0_arm64.whl (4.8 MB)
Using cached psycopg-3.3.6-py3-none-any.whl (215 kB)

[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip3.12 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [68]:
from sqlalchemy import create_engine, text
from getpass import getpass
from sqlalchemy.engine import URL

db_password = getpass("Enter PostgreSQL password: ")

connection_url = URL.create(
    drivername="postgresql+psycopg",
    username="postgres",
    password=db_password,
    host="localhost",
    port=5432,
    database="financial_analytics"
)

engine = create_engine(connection_url)

with engine.connect() as connection:
    result = connection.execute(text("SELECT version();"))
    print(result.scalar())

PostgreSQL 18.6 on aarch64-apple-darwin24.6.0, compiled by Apple clang version 17.0.0 (clang-1700.0.13.5), 64-bit


In [69]:
df_clean.to_sql(
    name="transactions_etl",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=1000,
    method="multi"
)

print("Transactions loaded successfully!")

Transactions loaded successfully!


In [70]:
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM transactions_etl;")
    )

    print("Rows in PostgreSQL:", result.scalar())

Rows in PostgreSQL: 10000


In [71]:
query = """
SELECT
    type,
    COUNT(*) AS transaction_count,
    SUM(amount) AS total_value,
    SUM("isFraud") AS fraud_count,
    ROUND(
        100.0 * SUM("isFraud") / COUNT(*),
        2
    ) AS fraud_rate_pct
FROM transactions_etl
GROUP BY type
ORDER BY fraud_rate_pct DESC;
"""

fraud_sql = pd.read_sql(query, engine)

fraud_sql

,type,transaction_count,total_value,fraud_count,fraud_rate_pct
0,TRANSFER,921,4.343938e+08,33.0,3.58
1,CASH_OUT,1321,2.292032e+08,35.0,2.65
2,DEBIT,344,1.230348e+06,0.0,0.00
3,PAYMENT,5465,4.312667e+07,0.0,0.00
4,CASH_IN,1949,3.275129e+08,0.0,0.00


In [72]:
kpi_query = """
SELECT
    COUNT(*) AS total_transactions,
    ROUND(SUM(amount)::numeric, 2) AS total_transaction_value,
    ROUND(AVG(amount)::numeric, 2) AS average_transaction_amount,
    SUM("isFraud") AS fraud_count,
    ROUND(
        100.0 * SUM("isFraud") / COUNT(*),
        2
    ) AS fraud_rate_pct
FROM transactions_etl;
"""

kpi_summary = pd.read_sql(kpi_query, engine)
kpi_summary

,total_transactions,total_transaction_value,average_transaction_amount,fraud_count,fraud_rate_pct
0,10000,1.035467e+09,103546.69,68.0,0.68


In [73]:
type_query = """
SELECT
    type,
    COUNT(*) AS transaction_count,
    ROUND(SUM(amount)::numeric, 2) AS total_value,
    ROUND(AVG(amount)::numeric, 2) AS average_amount
FROM transactions_etl
GROUP BY type
ORDER BY total_value DESC;
"""

type_summary = pd.read_sql(type_query, engine)
type_summary

,type,transaction_count,total_value,average_amount
0,TRANSFER,921,4.343938e+08,471654.52
1,CASH_IN,1949,3.275129e+08,168041.49
2,CASH_OUT,1321,2.292032e+08,173507.34
3,PAYMENT,5465,4.312667e+07,7891.43
4,DEBIT,344,1.230348e+06,3576.59


In [74]:
time_query = """
SELECT
    step,
    COUNT(*) AS transaction_count,
    SUM("isFraud") AS fraud_count,
    ROUND(
        100.0 * SUM("isFraud") / COUNT(*),
        2
    ) AS fraud_rate_pct
FROM transactions_etl
GROUP BY step
ORDER BY step;
"""

time_summary = pd.read_sql(time_query, engine)
time_summary.head()

,step,transaction_count,fraud_count,fraud_rate_pct
0,1,2708,16.0,0.59
1,2,1014,8.0,0.79
2,3,552,4.0,0.72
3,4,565,10.0,1.77
4,5,665,6.0,0.90


In [75]:
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

kpi_summary.to_csv(output_dir / "kpi_summary.csv", index=False)
type_summary.to_csv(output_dir / "type_summary.csv", index=False)
fraud_sql.to_csv(output_dir / "fraud_summary.csv", index=False)
time_summary.to_csv(output_dir / "time_summary.csv", index=False)